In [ ]:
!pip install pandas numpy scikit-learn folium -q

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving junction_risk_table.csv to junction_risk_table.csv


In [ ]:
import pandas as pd

risk_df = pd.read_csv(
    "junction_risk_table.csv"
)

risk_df.head()

,junction,incident_count,high_priority_ratio,road_closure_ratio,planned_event_ratio,unique_causes,cluster,risk_level
0,17th Mn 1st Crs Aishwarya Stores Jn,1,0.000000,0.000000,0.000000,1,3,Medium
1,27th Cross Jayanagar(Ganapathi Temple),1,0.000000,0.000000,0.000000,1,3,Medium
2,28thMainJayanagarJunc,6,0.166667,0.000000,0.000000,1,3,Medium
3,29thMainRdBTM LayoutJunc,5,0.000000,0.000000,0.000000,3,3,Medium
4,5thMainHSR,3,1.000000,0.333333,0.333333,3,2,Low


In [ ]:
uploaded = files.upload()

Saving Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv to Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv


In [ ]:
event_file = list(uploaded.keys())[0]

event_df = pd.read_csv(
    event_file,
    engine="python",
    on_bad_lines="skip"
)

print(event_df.shape)

(8173, 46)


In [ ]:
junction_coords = (

    event_df

    .groupby("junction")

    .agg({

        "latitude":"mean",

        "longitude":"mean"

    })

    .reset_index()

)

junction_coords.head()

,junction,latitude,longitude
0,17th Mn 1st Crs Aishwarya Stores Jn,12.925044,77.633626
1,27th Cross Jayanagar(Ganapathi Temple),12.931752,77.579739
2,28thMainJayanagarJunc,12.916833,77.595374
3,29thMainRdBTM LayoutJunc,12.916210,77.616143
4,5thMainHSR,12.916451,77.631317


In [ ]:
print(risk_df.columns)

Index(['junction', 'incident_count', 'high_priority_ratio',
       'road_closure_ratio', 'planned_event_ratio', 'unique_causes', 'cluster',
       'risk_level'],
      dtype='object')


In [ ]:
junction_data = junction_coords.merge(

    risk_df[
        [
            "junction",
            "risk_level"
        ]
    ],

    on="junction",

    how="inner"

)

print(junction_data.shape)

junction_data.head()

(294, 4)


,junction,latitude,longitude,risk_level
0,17th Mn 1st Crs Aishwarya Stores Jn,12.925044,77.633626,Medium
1,27th Cross Jayanagar(Ganapathi Temple),12.931752,77.579739,Medium
2,28thMainJayanagarJunc,12.916833,77.595374,Medium
3,29thMainRdBTM LayoutJunc,12.916210,77.616143,Medium
4,5thMainHSR,12.916451,77.631317,Low


In [ ]:
from math import radians
from math import sin
from math import cos
from math import sqrt
from math import atan2

In [ ]:
def haversine(

    lat1,
    lon1,
    lat2,
    lon2

):

    R = 6371

    dlat = radians(
        lat2-lat1
    )

    dlon = radians(
        lon2-lon1
    )

    a = (

        sin(dlat/2)**2

        +

        cos(radians(lat1))

        *

        cos(radians(lat2))

        *

        sin(dlon/2)**2

    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1-a)
    )

    return R*c

In [ ]:
def suggest_diversion(

    affected_junction,

    top_n=3

):

    source = junction_data[

        junction_data["junction"]

        == affected_junction

    ]

    if len(source)==0:

        print(
            f"{affected_junction} not found"
        )

        return None

    lat = source.iloc[0]["latitude"]

    lon = source.iloc[0]["longitude"]

    candidates = junction_data[

        junction_data["risk_level"]

        .isin(

            ["Low","Medium"]

        )

    ].copy()

    candidates = candidates[
        candidates["junction"]
        != affected_junction
    ]

    candidates["distance"] = (

        candidates.apply(

            lambda row:

            haversine(

                lat,

                lon,

                row["latitude"],

                row["longitude"]

            ),

            axis=1

        )

    )

    candidates = (

        candidates

        .sort_values(
            "distance"
        )

        .head(top_n)

    )

    return candidates[
        [

            "junction",

            "latitude",

            "longitude",

            "risk_level",

            "distance"

        ]
    ]

In [ ]:
print(

    junction_data["junction"]

    .dropna()

    .sample(20)

)

236                       SagarHospitalJunc
96                          HSR14thMainJunc
149                        KrishnaFlourMill
140             KanakapuraRd-RingRdJunction
28                 BagalakunteCrossJunction
167    Malleswaram18thCrossRd-SampigeRdJunc
185                          N R SquareJunc
120                        Jaipuria-Adugodi
142                      KatriguppeJunction
213               PoliceTimmaiahCircle(GPO)
158                     LeprosyhospitalJunc
81                           ESI(k r puram)
89                           Geethanjali jn
228                      Richmond circle jn
219                      QueensStatueCircle
106                          HosmatJunction
211                      PlatformRdJunction
154          KuvempuCircleJunc,HAL MainGate
176              MillCornerRd-SampigeRdJunc
204       OldMadrasRd-SuddaguntepalyaRdJunc
Name: junction, dtype: object


In [ ]:
suggest_diversion(
    "Bommanahalli"
)

,junction,latitude,longitude,risk_level,distance
4,5thMainHSR,12.916451,77.631317,Low,1.108906
3,29thMainRdBTM LayoutJunc,12.916210,77.616143,Medium,1.659752
150,Krupanidhi College,12.924667,77.627637,Medium,1.969404


In [ ]:
import folium

In [ ]:
def diversion_map(

    affected_junction

):

    source = junction_data[

        junction_data["junction"]

        ==
        affected_junction

    ]

    if len(source)==0:

        print(
            f"{affected_junction} not found"
        )

        return None

    lat = source.iloc[0]["latitude"]

    lon = source.iloc[0]["longitude"]

    diversion = suggest_diversion(
        affected_junction
    )

    m = folium.Map(

        location=[
            lat,
            lon
        ],

        zoom_start=13

    )

    folium.Marker(

        [lat,lon],

        popup=
        f"""
        Affected Junction:
        {affected_junction}
        """,

        icon=folium.Icon(
            color="red"
        )

    ).add_to(m)

    for _,row in diversion.iterrows():

        folium.Marker(

            [

                row["latitude"],

                row["longitude"]

            ],

            popup=
            f"""
            Diversion:
            {row['junction']}

            Risk:
            {row['risk_level']}

            Distance:
            {round(row['distance'],2)} km
            """,

            icon=folium.Icon(
                color="green"
            )

        ).add_to(m)

    return m

In [ ]:
junction_name = input(
    "Enter affected junction: "
)

result = suggest_diversion(
    junction_name
)

print(result)

Enter affected junction: Bommanahalli
                     junction   latitude  longitude risk_level  distance
4                  5thMainHSR  12.916451  77.631317        Low  1.108906
3    29thMainRdBTM LayoutJunc  12.916210  77.616143     Medium  1.659752
150        Krupanidhi College  12.924667  77.627637     Medium  1.969404


In [ ]:
m = diversion_map(
    "Bommanahalli"
)

m

In [ ]:
all_diversions = []

for junction in junction_data["junction"]:

    try:

        alt = suggest_diversion(
            junction,
            top_n=3
        )

        if alt is not None:

            all_diversions.append({

                "affected_junction":
                junction,

                "alternative_1":
                alt.iloc[0]["junction"],

                "alternative_2":
                alt.iloc[1]["junction"],

                "alternative_3":
                alt.iloc[2]["junction"]

            })

    except:
        pass

diversion_df = pd.DataFrame(
    all_diversions
)

diversion_df.to_csv(
    "diversion_recommendations.csv",
    index=False
)

diversion_df.head()

,affected_junction,alternative_1,alternative_2,alternative_3
0,17th Mn 1st Crs Aishwarya Stores Jn,WiproJunc-Koramangala,Krupanidhi College,5thMainHSR
1,27th Cross Jayanagar(Ganapathi Temple),YediyurMeternityHospital,South end circle,KR Road-14thCross Junc
2,28thMainJayanagarJunc,MarenahalliRd-18thMainRdJunc,Arbindo Circle,JP Nagar 9th cross-24th main jn
3,29thMainRdBTM LayoutJunc,BTM16thMain-ORR Junc,Krupanidhi College,5thMainHSR
4,5thMainHSR,17th Mn 1st Crs Aishwarya Stores Jn,Krupanidhi College,WiproJunc-Koramangala


In [ ]:
result = suggest_diversion(
    "Bommanahalli"
)

print(result)

                     junction   latitude  longitude risk_level  distance
4                  5thMainHSR  12.916451  77.631317        Low  1.108906
3    29thMainRdBTM LayoutJunc  12.916210  77.616143     Medium  1.659752
150        Krupanidhi College  12.924667  77.627637     Medium  1.969404


In [ ]:
all_distances = []

for j in junction_data["junction"]:

    try:

        alt = suggest_diversion(
            j,
            top_n=3
        )

        all_distances.extend(
            alt["distance"].tolist()
        )

    except:
        pass

print(
    "Average Diversion Distance:",
    round(
        sum(all_distances)/
        len(all_distances),
        2
    ),
    "km"
)

Average Diversion Distance: 1.66 km


In [ ]:
diversion_df.to_csv(
    "diversion_recommendations.csv",
    index=False
)